# Starting Preprocessing

In [1]:
# 0.0 Imports

import pandas as pd
import numpy as np

## Phase 1: Basic Cleanup

**GOAL:**
- to reduce the shape from (1783, 39) -> (1470, 28)

In [17]:
# 1.1 Import the raw dataset

df = pd.read_csv(r'D:\Hustle\Chennai-PG\Data\raw\chennai_pg_dataset.csv')
df.shape

(1783, 39)

In [18]:
# 1.2 Deduplication

"""
Drop duplicate rows corresponding to ID & OCCUPANCY
"""

df = df.drop_duplicates(subset=['id', 'occupancy'])
print(df.shape)
print(df.duplicated().sum())

(1621, 39)
0


In [19]:
# 1.3 Drop columns

"""
Drop columns that are not useful for modeling.

If columns is not provided, the default set of columns
identified during EDA will be removed.

col = ['id', 'title', 'address', 'total_bathrooms', 'warden', 'cooking_allowed', 'gate_closing_time', 'guardian_required', 'nonveg_allowed', 'smoking_allowed'] by phase 1 observation
col = ['lunch', 'breakfast', 'dinner'] by phase 3 observation
"""

df = df.drop(columns=['id', 'title', 'address', 'total_bathrooms', 'warden', 'cooking_allowed', 'gate_closing_time', 'guardian_required', 'nonveg_allowed', 'smoking_allowed', 'lunch', 'breakfast', 'dinner'])
df.shape

(1621, 26)

In [20]:
# 1.4 Drop rows

"""
Drop rows that are not useful for modelling.

- ~1% rows with missing values in key columns
- rent with 0 or NaNs
- occupancy is NaN
- Known confirmed correction for THIS dataset
"""

df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])
# Remove invalid/placeholder rents.
# The minimum realistic PG rent in Chennai is well above 1000.
df = df[df['rent'] >= 1000]
# Known confirmed correction for THIS dataset
df = df[~((df['deposit'] == 200000) & (df['rent'] == 25000) & (df['locality'] == 'Vadapalani'))]
df.shape


(1470, 26)

In [23]:
# 1.5 fix datatypes & renaming

"""
FIll missing boolean amenites with False
then convert the columns to bool type
"""

bool_cols = ['attached_bathroom', 'mess', 'wifi', 'laundry', 'power_backup',
        'refrigerator', 'common_tv', 'room_cleaning','room_ac', 
        'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding',
        'room_attached_bath',
]
df[bool_cols] = df[bool_cols].fillna(False).astype(bool)
df['parking'] = df['parking'].fillna('No Parking') # in EDA i actually replaced NaN as none, but it make sense to keep No Parking

df['available_for'] = df['available_for'].replace('Both', 'Anyone')


In [25]:
# 1.6 fixes for before doing imputations

# this is for fix the left influend skew-ness (should be done before imputation)
df['transit_score'] = df['transit_score'].replace(-10, np.nan)

# creating tag for msiing values rows
df['transit_score_missing'] = df['transit_score'].isna().astype(int)
df['lifestyle_score_missing'] = df['lifestyle_score'].isna().astype(int)

In [26]:
df.shape

(1470, 28)

## Phase 1 Observation

- our goal (**to reduce the shape from (1783, 39) -> (1470, 28)**) was satisfied